# Historical experiment: unweighted four-class baseline

This preliminary run reuses the earlier unigram TF-IDF vocabulary and split. It selects regularization by validation macro-F1. The archived validation result is **59.48% accuracy / 43.86% macro-F1**.

In [ ]:
"""Refit the original Spark TF-IDF multinomial LR baseline for four rating bands."""
import os, sys, json, time, csv
from pathlib import Path
from collections import Counter
import numpy as np
from scipy.sparse import csr_matrix
from scipy.special import softmax
from sklearn.metrics import classification_report, confusion_matrix
import hashlib

BASELINE_RUN = "rating_teammate_policy_20261106_155015_111148"
NAMES = ["low", "moderate", "good", "excellent"]

def read_data(project, train_limit):
    baseline = project / 'baseline_results' / BASELINE_RUN
    manifest = Counter()
    with (baseline / 'split_manifest.csv').open(encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            manifest[row['uniqueID'], row['text_group_id'], row['split']] += 1
    group_splits = {group: split for _, group, split in manifest}

    data = {'train': [], 'validation': [], 'test': []}
    with (project / 'clean_data.csv').open(encoding='utf-8-sig', newline='') as f:
        for row in csv.DictReader(f):
            tokens = json.loads(row['filtered_words'])
            key = hashlib.sha256(f"{row['review']}|{row['rating']}".encode()).hexdigest()
            canonical = json.dumps(tokens, ensure_ascii=False, separators=(',', ':'))
            group = hashlib.sha256(canonical.encode()).hexdigest()
            split = group_splits.get(group)
            entry = (key, group, split)
            if not manifest[entry]:
                raise ValueError(f'Token sequence or row differs from manifest for {key}')
            manifest[entry] -= 1
            data[split].append((tokens, int(row['rating']) - 1))
    if manifest.total():
        raise ValueError(f'{manifest.total()} manifest rows missing from the data')
    if train_limit and train_limit < len(data['train']):
        indices = np.random.default_rng(SEED).choice(len(data['train']), train_limit, replace=False)
        data['train'] = [data['train'][index] for index in sorted(indices)]
    return data

def band(zero_based_rating):
    assert 0 <= zero_based_rating <= 9
    return 0 if zero_based_rating <= 2 else 1 if zero_based_rating <= 5 else 2 if zero_based_rating <= 7 else 3

OUT = Path.cwd()/'four_class_results'/'baseline'

def features(rows,vocabulary,idf):
    vocab = {word:i for i,word in enumerate(vocabulary)}
    indptr,indices,values = [0],[],[]
    for words,_ in rows:
        counts = Counter(vocab[w] for w in words if w in vocab)
        for i,n in sorted(counts.items()):
            indices.append(i)
            values.append(n*idf[i])
        indptr.append(len(indices))
    return csr_matrix((values,indices,indptr),shape=(len(rows),len(vocab)))

def evaluate(target,probability):
    predicted = probability.argmax(1)
    report = classification_report(target,predicted,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
    return dict(accuracy=float((target==predicted).mean()),macro_f1=report['macro avg']['f1-score'],
                precision_macro=report['macro avg']['precision'],recall_macro=report['macro avg']['recall'],
                classification_report=report,confusion_matrix=confusion_matrix(target,predicted,labels=range(4)).tolist(),
                predicted_counts=np.bincount(predicted,minlength=4).tolist())

def main():
    OUT.mkdir(parents=True,exist_ok=True)
    data = read_data(Path.cwd(), 0)
    ref = Path.cwd()/'baseline_results'/BASELINE_RUN
    vocabulary = json.loads((ref/'feature_metadata.json').read_text(encoding='utf-8'))['vocabulary']
    idf = np.load(ref/'idf_values.npz')['idf']
    matrices = {s:features(data[s],vocabulary,idf) for s in ('train','validation')}
    targets = {s:np.array([band(y) for _,y in data[s]]) for s in matrices}
    # Verify reconstructed features against the saved original classifier before refitting.
    old = np.load(ref/'review_only_reg_0p1_parameters.npz')
    original_prediction = softmax(matrices['validation']@old['coefficients'].T+old['intercept'],axis=1).argmax(1)
    accuracy = float((original_prediction==np.array([y for _,y in data['validation']])).mean())
    with (ref/'rating_model_comparison.csv').open() as f:
        previous = next(r for r in csv.DictReader(f) if r['model']=='review_only_reg_0p1')
    print('Feature check observed/reference',accuracy,previous['validation_classification_accuracy'],flush=True)
    assert np.isclose(accuracy,float(previous['validation_classification_accuracy']),atol=1e-10,rtol=0)
    print('Original baseline feature reconstruction verified:',accuracy,flush=True)
    config = dict(model='Spark multinomial LogisticRegression',spark='4.2.0',reg_params=[0.1,0.3],
                  max_iter=150,tol=1e-6,standardization=True,elastic_net_param=0.0,
                  vocabulary_n=len(vocabulary),feature_source=str(ref),feature_normalization='none; raw counts times saved Spark IDF',
                  original_feature_reproduction_accuracy=accuracy,train_n=len(targets['train']),validation_n=len(targets['validation']),
                  selection='best validation Macro-F1; tie stronger regularization',class_weighting=False,test_evaluated=False,
                  labels={'low':[1,2,3],'moderate':[4,5,6],'good':[7,8],'excellent':[9,10]})
    (OUT/'config.json').write_text(json.dumps(config,indent=2))
    os.environ['PYSPARK_PYTHON']=sys.executable
    os.environ['PYSPARK_DRIVER_PYTHON']=sys.executable
    from pyspark.sql import SparkSession
    from pyspark.ml.linalg import Vectors, VectorUDT
    from pyspark.sql.types import StructType,StructField,DoubleType
    from pyspark.ml.classification import LogisticRegression
    spark=(SparkSession.builder.master('local[2]').appName('FourClassBaseline')
           .config('spark.ui.enabled','false').config('spark.sql.shuffle.partitions','2')
           .config('spark.pyspark.python',sys.executable).config('spark.pyspark.driver.python',sys.executable)
           .getOrCreate())
    spark.sparkContext.setLogLevel('ERROR')
    schema=StructType([StructField('label',DoubleType(),False),StructField('features',VectorUDT(),False)])
    matrix=matrices['train']
    rows=[(float(y),Vectors.sparse(matrix.shape[1],matrix.indices[matrix.indptr[i]:matrix.indptr[i+1]],matrix.data[matrix.indptr[i]:matrix.indptr[i+1]])) for i,y in enumerate(targets['train'])]
    train=spark.createDataFrame(rows,schema).repartition(2).cache()
    assert train.count()==72108
    results=[]
    try:
        for reg in (0.1,0.3):
            start=time.perf_counter()
            print('Training baseline reg',reg,flush=True)
            model=LogisticRegression(featuresCol='features',labelCol='label',family='multinomial',
                regParam=reg,elasticNetParam=0.0,maxIter=150,standardization=True,tol=1e-6).fit(train)
            coefficients=model.coefficientMatrix.toArray()
            intercept=model.interceptVector.toArray()
            probability=softmax(matrices['validation']@coefficients.T+intercept,axis=1)
            result=dict(model='Baseline LR',embedding='TF-IDF',reg_param=reg,iterations=int(model.summary.totalIterations),
                        hit_iteration_limit=int(model.summary.totalIterations)>=150,seconds=time.perf_counter()-start,
                        objective_history=list(model.summary.objectiveHistory),**evaluate(targets['validation'],probability))
            # Compare exported-parameter probabilities with Spark's own prediction path.
            check=matrices['validation'][:20]
            check_rows=[(float(i),Vectors.sparse(check.shape[1],check.indices[check.indptr[i]:check.indptr[i+1]],check.data[check.indptr[i]:check.indptr[i+1]])) for i in range(20)]
            transformed=model.transform(spark.createDataFrame(check_rows,schema)).select('label','probability').collect()
            for row in transformed:
                assert np.allclose(row.probability.toArray(),probability[int(row.label)],atol=1e-10,rtol=0)
            stem=f'lr_reg_{reg:g}'
            np.savez_compressed(OUT/f'{stem}_parameters.npz',coefficients=coefficients,intercept=intercept)
            np.savez_compressed(OUT/f'{stem}_validation.npz',target=targets['validation'],probability=probability)
            (OUT/f'{stem}.json').write_text(json.dumps(result,indent=2))
            results.append(result)
            (OUT/'summary.json').write_text(json.dumps(results,indent=2))
            print(stem,{k:result[k] for k in ('accuracy','macro_f1','precision_macro','recall_macro','iterations')},flush=True)
        best=max(results,key=lambda r:(r['macro_f1'],r['reg_param']))
        (OUT/'best.json').write_text(json.dumps(best,indent=2))
        print('Baseline completed and verified',flush=True)
    finally:
        spark.stop()

In [ ]:
# Run with the original split and saved TF-IDF artifacts in the repository root.
main()